# Dynamic FEA vs. Multimodal Significance Tests

This notebook compares the dynamic FEA-based and multimodal FER models.

The test set contains **378 reenactments**. Each reenactment contributes one FEA prediction and two multimodal predictions, one for the central-view image and one for the side-view image. The two multimodal samples belonging to the same reenactment are therefore not treated as independent experimental units.

## Analysis plan

1. Primary comparison: compare pooled multimodal accuracy with FEA accuracy using a paired cluster bootstrap over reenactments.
2. Sensitivity check: test the same reenactment-level mean difference with a one-sample t-test.
3. Secondary analysis: run exact McNemar tests for FEA vs. Central Multimodal, FEA vs. Side Multimodal, and Central vs. Side Multimodal, followed by Holm correction.

### Note on the primary resampling test

A simple paired label-permutation/sign-flip test is not used because the reenactment-level multimodal score has support $\{0, 0.5, 1\}$, whereas FEA correctness has support $\{0, 1\}$. Exchanging the two measurements within a pair would therefore require an exchangeability assumption that does not hold by construction.

Instead, the primary significance test uses a **centered paired cluster bootstrap** on the reenactment-level accuracy differences. The same clustering scheme is used to obtain the 95% confidence interval for the accuracy difference.


## 1. Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import ttest_1samp
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests


PREDICTIONS_PATH = Path("dynamic_test_predictions.csv")

RANDOM_SEED = 42

# Use many resamples for stable final confidence intervals and p-values.
# Resampling is processed in batches below to keep memory usage low.
N_BOOTSTRAP = 1_000_000
BOOTSTRAP_BATCH_SIZE = 10_000

ALPHA = 0.05


## 2. Load and Validate Predictions

The prediction CSV is expected to contain one row per image-view sample and the columns 

-`sample_id`
- `reenactment_id`
- `camera_index`
- `true_label_id`
- `fea_pred_id`
- `multimodal_pred_id`

Camera index `0` denotes the central view and camera index `1` the side view.


In [2]:
prediction_df = pd.read_csv(PREDICTIONS_PATH)

print(f"Rows: {len(prediction_df)}")
print(f"Columns: {len(prediction_df.columns)}")
prediction_df.head()


Rows: 756
Columns: 39


,sample_id,reenactment_id,timestamp,set_id,participant_id,level_id,emoji_id,camera_index,perspective,true_label_id,...,fea_prob_surprise,multimodal_pred_id,multimodal_pred,multimodal_prob_anger,multimodal_prob_disgust,multimodal_prob_fear,multimodal_prob_happiness,multimodal_prob_neutral,multimodal_prob_sadness,multimodal_prob_surprise
0,1700478995850-2-1-1-0-0-0,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,0,Central,0,...,0.016200,0,Anger,0.798008,0.054746,0.005280,0.006042,0.009937,0.117664,0.008323
1,1700478995850-2-1-1-0-0-1,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,1,Side,0,...,0.016200,0,Anger,0.586113,0.127385,0.010630,0.006146,0.022720,0.239457,0.007548
2,1700478998549-2-1-1-1-5-0,1700478998549-2-1-1-1-5,1700478998549,2,1,1,1,0,Central,5,...,0.000089,5,Sadness,0.016788,0.003464,0.003549,0.006522,0.007196,0.958934,0.003547
3,1700478998549-2-1-1-1-5-1,1700478998549-2-1-1-1-5,1700478998549,2,1,1,1,1,Side,5,...,0.000089,5,Sadness,0.016804,0.003480,0.003540,0.006520,0.007197,0.958908,0.003550
4,1700479001137-2-1-1-2-3-0,1700479001137-2-1-1-2-3,1700479001137,2,1,1,2,0,Central,3,...,0.000331,3,Happiness,0.002462,0.004623,0.003282,0.982724,0.001283,0.002237,0.003387


In [3]:
required_columns = {
    "sample_id",
    "reenactment_id",
    "participant_id",
    "camera_index",
    "true_label_id",
    "fea_pred_id",
    "multimodal_pred_id"
}

missing_columns = required_columns - set(prediction_df.columns)
assert not missing_columns, f"Missing required columns: {sorted(missing_columns)}"

assert len(prediction_df) == 756
assert prediction_df["sample_id"].is_unique
assert prediction_df["reenactment_id"].nunique() == 378
assert set(prediction_df["camera_index"].unique()) == {0, 1}

samples_per_reenactment = prediction_df.groupby("reenactment_id").size()
assert samples_per_reenactment.eq(2).all()

views_per_reenactment = prediction_df.groupby("reenactment_id")["camera_index"].nunique()
assert views_per_reenactment.eq(2).all()

true_labels_per_reenactment = prediction_df.groupby("reenactment_id")["true_label_id"].nunique()
assert true_labels_per_reenactment.eq(1).all()

participants_per_reenactment = prediction_df.groupby("reenactment_id")["participant_id"].nunique()
assert participants_per_reenactment.eq(1).all()
assert prediction_df["participant_id"].nunique() == 8

fea_predictions_per_reenactment = prediction_df.groupby("reenactment_id")["fea_pred_id"].nunique()
assert fea_predictions_per_reenactment.eq(1).all()

print("Prediction-table structure validated.")


Prediction-table structure validated.


## 3. Construct Reenactment-Level Analysis Table

For each reenactment, $M_{C,i}$ indicates whether the central-view multimodal prediction was correct, $M_{S,i}$ indicates whether the side-view multimodal prediction was correct, and $F_i$ indicates whether the FEA prediction was correct.

The pooled multimodal contribution of reenactment $i$ is $M_i = \frac{M_{C,i} + M_{S,i}}{2}$.

Averaging $M_i$ across all 378 reenactments exactly reproduces the pooled multimodal accuracy over all 756 image-view samples.


In [4]:
prediction_df = prediction_df.copy()

prediction_df["multimodal_correct"] = prediction_df["multimodal_pred_id"] == prediction_df["true_label_id"]
prediction_df["fea_correct"] = prediction_df["fea_pred_id"] == prediction_df["true_label_id"]


In [5]:
multimodal_correct_by_view = (
    prediction_df
    .pivot(index="reenactment_id", columns="camera_index", values="multimodal_correct")
    .rename(columns={0: "central_multimodal_correct", 1: "side_multimodal_correct"})
)

fea_correct = prediction_df.groupby("reenactment_id")["fea_correct"].first()
participant_id = prediction_df.groupby("reenactment_id")["participant_id"].first()

analysis_df = multimodal_correct_by_view.join(fea_correct).join(participant_id)

analysis_df["central_multimodal_correct"] = analysis_df["central_multimodal_correct"].astype(bool)
analysis_df["side_multimodal_correct"] = analysis_df["side_multimodal_correct"].astype(bool)
analysis_df["fea_correct"] = analysis_df["fea_correct"].astype(bool)

analysis_df["multimodal_correct_mean"] = (
    analysis_df["central_multimodal_correct"].astype(float) + analysis_df["side_multimodal_correct"].astype(float)
) / 2.0

assert len(analysis_df) == 378
assert not analysis_df.isna().any().any()
assert set(analysis_df["multimodal_correct_mean"].unique()).issubset({0.0, 0.5, 1.0})

analysis_df.head()


,central_multimodal_correct,side_multimodal_correct,fea_correct,participant_id,multimodal_correct_mean
reenactment_id,,,,,
1700478995850-2-1-1-0-0,True,True,True,1,1.0
1700478998549-2-1-1-1-5,True,True,True,1,1.0
1700479001137-2-1-1-2-3,True,True,True,1,1.0
1700479004312-2-1-1-3-0,False,False,False,1,0.0
1700479005401-2-1-1-4-0,True,True,True,1,1.0


## 4. Verify Reported Performance

In [6]:
pooled_multimodal_accuracy = analysis_df["multimodal_correct_mean"].mean()
fea_accuracy = analysis_df["fea_correct"].mean()

central_multimodal_accuracy = analysis_df["central_multimodal_correct"].mean()
side_multimodal_accuracy = analysis_df["side_multimodal_correct"].mean()

accuracy_difference = pooled_multimodal_accuracy - fea_accuracy

print(f"Pooled multimodal accuracy: {pooled_multimodal_accuracy:.4%}")
print(f"FEA accuracy:               {fea_accuracy:.4%}")
print(f"Central multimodal accuracy:{central_multimodal_accuracy: .4%}")
print(f"Side multimodal accuracy:   {side_multimodal_accuracy:.4%}")
print(f"Multimodal - FEA:           {100 * accuracy_difference:.2f} percentage points")

assert np.isclose(pooled_multimodal_accuracy, prediction_df["multimodal_correct"].mean())
assert np.isclose(fea_accuracy, prediction_df["fea_correct"].mean())

assert prediction_df["multimodal_correct"].sum() == 617
assert np.isclose(pooled_multimodal_accuracy, 617 / 756)

assert prediction_df["fea_correct"].sum() == 592
assert analysis_df["fea_correct"].sum() == 296
assert np.isclose(fea_accuracy, 296 / 378)


Pooled multimodal accuracy: 81.6138%
FEA accuracy:               78.3069%
Central multimodal accuracy: 81.7460%
Side multimodal accuracy:   81.4815%
Multimodal - FEA:           3.31 percentage points


## 5. Primary FEA vs. Multimodal Comparison

The primary estimand is the difference between the reported overall accuracies, $\Delta = \mathrm{Accuracy}_{Multimodal} - \mathrm{Accuracy}_{FEA}$.

For each reenactment, $d_i = M_i - F_i$. The 378 reenactment-level differences are used as the resampling units.

The analysis estimates:

1. a two-sided bootstrap p-value for $H_0: \Delta = 0$, using the centered empirical distribution under the null
2. a percentile-bootstrap 95% confidence interval for $\Delta$

Positive values favor the multimodal model.


In [7]:
def paired_cluster_bootstrap(differences: np.ndarray,
                             n_bootstrap: int = N_BOOTSTRAP,
                             batch_size: int = BOOTSTRAP_BATCH_SIZE,
                             seed: int = RANDOM_SEED,
                             alpha: float = ALPHA) -> dict:
    
    differences = np.asarray(differences, dtype=float)

    if differences.ndim != 1 or differences.size == 0:
        raise ValueError("differences must be a nonempty one-dimensional array.")
    if not np.isin(differences, [-1.0, -0.5, 0.0, 0.5, 1.0]).all():
        raise ValueError("Expected paired accuracy differences in {-1, -0.5, 0, 0.5, 1}.")

    n = len(differences)
    observed_difference = differences.mean()

    doubled_differences = (2 * differences).astype(np.int64)
    observed_sum = doubled_differences.sum()

    rng = np.random.default_rng(seed)
    bootstrap_means = np.empty(n_bootstrap, dtype=float)
    n_extreme = 0

    for start in range(0, n_bootstrap, batch_size):
        end = min(start + batch_size, n_bootstrap)
        current_batch_size = end - start

        # Ordinary paired bootstrap for the confidence interval.
        bootstrap_indices = rng.integers(0, n, size=(current_batch_size, n))
        bootstrap_means[start:end] = differences[bootstrap_indices].mean(axis=1)

        # Centered-bootstrap null test evaluated in exact integer arithmetic:
        # |mean(d*) - mean(d)| >= |mean(d)| is equivalent to |S* - S| >= |S|,
        # where q_i = 2 d_i, S = sum(q_i), and S* = sum(q_i*).
        null_indices = rng.integers(0, n, size=(current_batch_size, n))
        null_sums = doubled_differences[null_indices].sum(axis=1)

        n_extreme += np.count_nonzero(
            np.abs(null_sums - observed_sum) >= abs(observed_sum)
        )

    ci_low, ci_high = np.quantile(bootstrap_means, [alpha / 2, 1 - alpha / 2])
    p_value = (n_extreme + 1) / (n_bootstrap + 1)

    return {
        "n": n,
        "difference": observed_difference,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "p_value": p_value,
        "n_bootstrap": n_bootstrap
    }


In [8]:
reenactment_differences = (
    analysis_df["multimodal_correct_mean"] - analysis_df["fea_correct"].astype(float)
).to_numpy()

primary_result = paired_cluster_bootstrap(
    differences=reenactment_differences,
    n_bootstrap=N_BOOTSTRAP,
    batch_size=BOOTSTRAP_BATCH_SIZE,
    seed=RANDOM_SEED,
    alpha=ALPHA
)

primary_result_df = pd.DataFrame([{
    "comparison": "Multimodal - FEA",
    "n_reenactments": primary_result["n"],
    "fea_accuracy": fea_accuracy,
    "multimodal_accuracy": pooled_multimodal_accuracy,
    "difference_pp": 100 * primary_result["difference"],
    "ci_low_pp": 100 * primary_result["ci_low"],
    "ci_high_pp": 100 * primary_result["ci_high"],
    "p_value": primary_result["p_value"],
    "n_bootstrap": primary_result["n_bootstrap"]
}])

primary_result_df


,comparison,n_reenactments,fea_accuracy,multimodal_accuracy,difference_pp,ci_low_pp,ci_high_pp,p_value,n_bootstrap
0,Multimodal - FEA,378,0.783069,0.816138,3.306878,0.26455,6.481481,0.043552,1000000


In [9]:
row = primary_result_df.iloc[0]

print(f"Multimodal - FEA accuracy difference: {row['difference_pp']:.2f} percentage points")
print(f"{100 * (1 - ALPHA):.0f}% bootstrap CI: [{row['ci_low_pp']:.2f}, {row['ci_high_pp']:.2f}] percentage points")
print(f"Two-sided bootstrap p-value: {row['p_value']:.12f}")


Multimodal - FEA accuracy difference: 3.31 percentage points
95% bootstrap CI: [0.26, 6.48] percentage points
Two-sided bootstrap p-value: 0.043551956448


## 6. Sensitivity Check

As a sensitivity analysis, the same 378 reenactment-level differences $d_i = M_i - F_i$ are tested with a one-sample t-test against a mean of zero.

This is not a separate primary hypothesis test. It checks whether a conventional parametric test leads to the same substantive conclusion as the bootstrap analysis. No multiplicity correction is applied to this sensitivity check.


In [10]:
sensitivity_test = ttest_1samp(reenactment_differences, popmean=0)

sensitivity_result_df = pd.DataFrame([{
    "comparison": "Multimodal - FEA",
    "n_reenactments": len(reenactment_differences),
    "difference_pp": 100 * reenactment_differences.mean(),
    "t_statistic": sensitivity_test.statistic,
    "degrees_of_freedom": sensitivity_test.df,
    "p_value": sensitivity_test.pvalue
}])

sensitivity_result_df


,comparison,n_reenactments,difference_pp,t_statistic,degrees_of_freedom,p_value
0,Multimodal - FEA,378,3.306878,2.056813,377,0.040392


In [11]:
row = sensitivity_result_df.iloc[0]

print(f"Mean difference: {row['difference_pp']:.2f} percentage points")
print(f"t({row['degrees_of_freedom']:.0f}) = {row['t_statistic']:.3f}")
print(f"Two-sided sensitivity-check p-value: {row['p_value']:.12f}")


Mean difference: 3.31 percentage points
t(377) = 2.057
Two-sided sensitivity-check p-value: 0.040391842702


## 7. Secondary View-Specific Comparisons

The secondary analysis uses three exact paired McNemar tests over the same 378 reenactments:

1. FEA vs. Central Multimodal
2. FEA vs. Side Multimodal
3. Central Multimodal vs. Side Multimodal

For two paired binary correctness variables $A$ and $B$, only discordant pairs contribute to McNemar's test. Because the three pairwise questions are tested simultaneously, their p-values are corrected using the Holm method.


In [12]:
def exact_mcnemar(df: pd.DataFrame, column_a: str, column_b: str, label_a: str, label_b: str) -> dict:
    a = df[column_a].astype(bool).to_numpy()
    b = df[column_b].astype(bool).to_numpy()

    both_correct = int(np.sum(a & b))
    a_only = int(np.sum(a & ~b))
    b_only = int(np.sum(~a & b))
    both_wrong = int(np.sum(~a & ~b))

    table = np.array([[both_correct, a_only], [b_only, both_wrong]])
    result = mcnemar(table, exact=True, correction=False)

    return {
        "comparison": f"{label_a} vs. {label_b}",
        "n": len(df),
        "accuracy_a": a.mean(),
        "accuracy_b": b.mean(),
        "difference_b_minus_a_pp": 100 * (b.mean() - a.mean()),
        "both_correct": both_correct,
        "a_only_correct": a_only,
        "b_only_correct": b_only,
        "both_wrong": both_wrong,
        "p_raw": result.pvalue
    }


In [13]:
secondary_results = [
    exact_mcnemar(analysis_df, "fea_correct", "central_multimodal_correct", "FEA", "Central Multimodal"),
    exact_mcnemar(analysis_df, "fea_correct", "side_multimodal_correct", "FEA", "Side Multimodal"),
    exact_mcnemar(analysis_df, "central_multimodal_correct", "side_multimodal_correct", "Central Multimodal", "Side Multimodal")
]

secondary_result_df = pd.DataFrame(secondary_results)

reject, p_holm, _, _ = multipletests(secondary_result_df["p_raw"], alpha=ALPHA, method="holm")
secondary_result_df["p_holm"] = p_holm
secondary_result_df["significant_holm"] = reject

secondary_result_df


,comparison,n,accuracy_a,accuracy_b,difference_b_minus_a_pp,both_correct,a_only_correct,b_only_correct,both_wrong,p_raw,p_holm,significant_holm
0,FEA vs. Central Multimodal,378,0.783069,0.817460,3.439153,281,15,28,54,0.065994,0.197982,False
1,FEA vs. Side Multimodal,378,0.783069,0.814815,3.174603,279,17,29,53,0.103805,0.207611,False
2,Central Multimodal vs. Side Multimodal,378,0.817460,0.814815,-0.264550,294,15,14,55,1.000000,1.000000,False


## 8. Participant-Level Heterogeneity Check

This descriptive check examines whether the pooled Multimodal-vs.-FEA difference is directionally consistent across the eight test participants or is mainly driven by a small number of participants.

For each participant, the table reports the number of reenactments, FEA accuracy, pooled multimodal accuracy, and the difference $\mathrm{Accuracy}_{Multimodal} - \mathrm{Accuracy}_{FEA}$.

No participant-level significance test or correction factor is applied. The primary inference remains the reenactment-level analysis above; this section is a heterogeneity and plausibility check.


In [14]:
participant_result_df = (
    analysis_df.reset_index()
    .groupby("participant_id", as_index=False)
    .agg(
        n_reenactments=("reenactment_id", "size"),
        fea_accuracy=("fea_correct", "mean"),
        multimodal_accuracy=("multimodal_correct_mean", "mean")
    )
)

participant_result_df["difference_pp"] = 100 * (
    participant_result_df["multimodal_accuracy"] - participant_result_df["fea_accuracy"]
)

n_multimodal_better = int((participant_result_df["difference_pp"] > 0).sum())
n_fea_better = int((participant_result_df["difference_pp"] < 0).sum())
n_equal = int((participant_result_df["difference_pp"] == 0).sum())
median_participant_difference_pp = participant_result_df["difference_pp"].median()

print(f"Participants favoring Multimodal: {n_multimodal_better}/8")
print(f"Participants favoring FEA:        {n_fea_better}/8")
print(f"Participants tied:                 {n_equal}/8")
print(f"Median participant difference (Multimodal - FEA): {median_participant_difference_pp:.2f} percentage points")

participant_result_df


Participants favoring Multimodal: 4/8
Participants favoring FEA:        4/8
Participants tied:                 0/8
Median participant difference (Multimodal - FEA): 1.86 percentage points


,participant_id,n_reenactments,fea_accuracy,multimodal_accuracy,difference_pp
0,1,47,0.723404,0.861702,13.829787
1,8,54,0.722222,0.703704,-1.851852
2,10,46,0.804348,0.728261,-7.608696
3,13,46,0.934783,0.891304,-4.347826
4,15,48,0.625000,0.781250,15.625000
5,18,43,0.976744,0.965116,-1.162791
6,23,53,0.792453,0.858491,6.603774
7,27,41,0.707317,0.756098,4.878049


## 9. Summary and Export

The primary result answers whether the reported pooled dynamic multimodal accuracy differs from the dynamic FEA accuracy while respecting the reenactment-level dependency structure. The one-sample t-test provides a sensitivity check of the same mean difference. The secondary McNemar tests determine whether the multimodal advantage differs by camera perspective and whether central and side multimodal samples differ in classification accuracy. The participant-level breakdown is descriptive and is used to assess directional consistency and heterogeneity across the eight test participants.


In [15]:
summary_df = pd.DataFrame({
    "metric": [
        "FEA accuracy",
        "Pooled multimodal accuracy",
        "Central multimodal accuracy",
        "Side multimodal accuracy",
        "Multimodal - FEA difference (pp)",
        "Primary bootstrap CI low (pp)",
        "Primary bootstrap CI high (pp)",
        "Primary bootstrap p-value",
        "Sensitivity t statistic",
        "Sensitivity t-test p-value",
        "Participants favoring Multimodal",
        "Participants favoring FEA",
        "Participants tied",
        "Median participant difference Multimodal - FEA (pp)"
    ],
    "value": [
        fea_accuracy,
        pooled_multimodal_accuracy,
        central_multimodal_accuracy,
        side_multimodal_accuracy,
        100 * primary_result["difference"],
        100 * primary_result["ci_low"],
        100 * primary_result["ci_high"],
        primary_result["p_value"],
        sensitivity_test.statistic,
        sensitivity_test.pvalue,
        n_multimodal_better,
        n_fea_better,
        n_equal,
        median_participant_difference_pp
    ]
})

summary_df


,metric,value
0,FEA accuracy,0.783069
1,Pooled multimodal accuracy,0.816138
2,Central multimodal accuracy,0.817460
3,Side multimodal accuracy,0.814815
4,Multimodal - FEA difference (pp),3.306878
5,Primary bootstrap CI low (pp),0.264550
6,Primary bootstrap CI high (pp),6.481481
7,Primary bootstrap p-value,0.043552
8,Sensitivity t statistic,2.056813
9,Sensitivity t-test p-value,0.040392


In [16]:
OUTPUT_DIR = Path("statistical_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

analysis_df.to_csv(OUTPUT_DIR / "dynamic_fea_vs_multimodal_reenactment_table.csv", index=True)
primary_result_df.to_csv(OUTPUT_DIR / "dynamic_fea_vs_multimodal_primary_result.csv", index=False)
sensitivity_result_df.to_csv(OUTPUT_DIR / "dynamic_fea_vs_multimodal_sensitivity_ttest.csv", index=False)
secondary_result_df.to_csv(OUTPUT_DIR / "dynamic_fea_vs_multimodal_secondary_mcnemar_results.csv", index=False)
participant_result_df.to_csv(OUTPUT_DIR / "dynamic_fea_vs_multimodal_participant_level_results.csv", index=False)
summary_df.to_csv(OUTPUT_DIR / "dynamic_fea_vs_multimodal_summary.csv", index=False)

print(f"Results written to: {OUTPUT_DIR.resolve()}")


Results written to: /workspace/repos/emohevrdb-dfer/6_discussion/significance-tests/dynamic-significance-tests/statistical_results
